# FPS Evaluation Workflow: All Methods on Mip-NeRF 360 & Synthetic Specular

This notebook automates the **rendering FPS evaluation** across 4 methods:
1. **3DGS** (using `gaussian-splatting_backup/render.py`)
2. **FastGS** (using `FastGS_backup_v2/render.py`)
3. **Specular-Gaussians** (using `Specular-Gaussians_backup_v2/render.py`)
4. **Spec-FastGS (Ours)** (using `spec-fastgs/render.py`)

### Workflow Steps:
- **Step 1**: Authenticate & download pre-trained point clouds from Hugging Face datasets with HF Token & Proxy bypass.
- **Step 2**: Extract zip archives (`images4.zip`, `result.zip`, `synthetic_specular.zip`).
- **Step 3**: Execute CUDA-synchronized rendering benchmarks (`render.py`) for each scene.
- **Step 4**: Parse `results.json`, aggregate FPS metrics, and display summary tables.


In [ ]:
import os
import sys
import ssl
import json
import glob
import time
import zipfile
import warnings
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# 1. Ensure UTF-8 console output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# 2. Load Hugging Face Token (from environment variable or hf_token.json)
BASE_DIR = Path(__file__).parent.parent if '__file__' in locals() else Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
token_file = BASE_DIR / "hf_token.json"

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and token_file.exists():
    try:
        with open(token_file, "r") as f:
            HF_TOKEN = json.load(f).get("HF_KEY")
    except Exception:
        pass

if not HF_TOKEN:
    print("[NOTICE] HF_TOKEN not found in environment or hf_token.json.")
    print("Please create 'hf_token.json' in workspace root with {'HF_KEY': 'your_token'}.")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN

# 3. Bypass SSL verification & handle Corporate Proxy for BOSCH / server networks
ssl._create_default_https_context = ssl._create_unverified_context
import requests
from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter('ignore', InsecureRequestWarning)
old_merge = requests.Session.merge_environment_settings
requests.Session.merge_environment_settings = lambda self, url, proxies, stream, verify, cert: \
    {**old_merge(self, url, proxies, stream, verify, cert), 'verify': False}

# 4. Verify huggingface_hub
try:
    from huggingface_hub import HfApi, hf_hub_download, login
except ImportError:
    print("Installing huggingface_hub...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi, hf_hub_download, login

if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("[SUCCESS] Authenticated with Hugging Face Hub successfully!")
    except Exception as e:
        print(f"[WARNING] HF Login warning: {e}")

STORAGE_DIR = BASE_DIR / "storage"
HF_CACHE_DIR = STORAGE_DIR / "hf_downloads"
RESULTS_ROOT = BASE_DIR / "result_mipnerf_specular"
KETQUA_DIR = BASE_DIR / "ketqua"

for d in [STORAGE_DIR, HF_CACHE_DIR, RESULTS_ROOT, KETQUA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Workspace Root : {BASE_DIR.resolve()}")
print(f"Active Device  : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# Dataset & Method Hugging Face Mapping
HF_DATASETS = {
    "Specular-Gaussians": {
        "mipnerf360": {
            "repo_id": "DiBiay/specular_gaussians-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "images4_spec_gaussian"
        },
        "specular": {
            "repo_id": "DiBiay/specular_gaussians-synthetic_specular-result",
            "filename": "synthetic_specular.zip",
            "target_dir": RESULTS_ROOT / "spec_gaussian_synthetic_specular"
        }
    },
    "Spec-FastGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/spec-fastgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "images4_spec_fastgs"
        },
        "specular": {
            "repo_id": "DiBiay/spec-fastgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "spec-fastgs-synthetic-specular"
        }
    },
    "FastGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/fastgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "images4_fastgs"
        },
        "specular": {
            "repo_id": "DiBiay/fastgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "fastgs-synthetic-specular"
        }
    },
    "3DGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/3dgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "images4_3dgs"
        },
        "specular": {
            "repo_id": "DiBiay/3dgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "3dgs-synthetic-specular"
        }
    }
}

CODEBASE_PATHS = {
    "3DGS": BASE_DIR / "gaussian-splatting_backup",
    "FastGS": BASE_DIR / "FastGS_backup_v2",
    "Specular-Gaussians": BASE_DIR / "Specular-Gaussians_backup_v2",
    "Spec-FastGS": BASE_DIR / "spec-fastgs"
}

SCENES_MIPNERF360 = ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]
SCENES_SPECULAR = ["ashtray", "dishes", "headphone", "jupyter", "lock", "plane", "record", "teapot"]

print("Hugging Face mapping & Codebases verified.")


In [ ]:
def download_and_extract_hf_dataset(method_name, dataset_type):
    cfg = HF_DATASETS[method_name][dataset_type]
    repo_id = cfg["repo_id"]
    filename = cfg["filename"]
    target_dir = Path(cfg["target_dir"])
    target_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n=== Downloading {method_name} ({dataset_type}) from HF: {repo_id}/{filename} ===")
    try:
        kwargs = {"repo_id": repo_id, "filename": filename, "repo_type": "dataset", "cache_dir": str(HF_CACHE_DIR)}
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN
            
        zip_path = hf_hub_download(**kwargs)
        print(f"Downloaded archive to: {zip_path}")
        
        print(f"Extracting archive into: {target_dir}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"[SUCCESS] Extracted {method_name} ({dataset_type}) successfully.")
        return True
    except Exception as e:
        print(f"[ERROR] Failed to download or extract {repo_id}/{filename}: {e}")
        return False

# Execute Download for all 4 Methods & Both Datasets
print("Starting Hugging Face dataset download sequence...")
for method in ["3DGS", "FastGS", "Specular-Gaussians", "Spec-FastGS"]:
    for ds_type in ["mipnerf360", "specular"]:
        download_and_extract_hf_dataset(method, ds_type)


In [ ]:
def find_model_scene_path(base_target_dir, scene_name):
    """Locates the exact model directory containing point_cloud or results.json."""
    p = Path(base_target_dir)
    candidates = [
        p / scene_name,
        p / f"{p.name}-result" / scene_name,
        p / "spec-fastgs-mipnerf360-result-images4" / scene_name,
        p / "spec-fastgs-synthetic-specular-result" / scene_name,
        p / "fastgs-mipnerf360-result" / scene_name,
        p / "fastgs-synthetic-specular-result" / scene_name,
        p / "3dgs-mipnerf360-result" / scene_name,
        p / "3dgs-synthetic-specular-result" / scene_name,
    ]
    for c in candidates:
        if c.exists() and ((c / "point_cloud").exists() or (c / "results.json").exists()):
            return c
    # Fallback glob search
    matches = list(p.rglob(scene_name))
    for m in matches:
        if m.is_dir() and ((m / "point_cloud").exists() or (m / "results.json").exists()):
            return m
    return p / scene_name

def run_fps_benchmark(method_name, dataset_type, scenes):
    code_dir = CODEBASE_PATHS[method_name]
    render_py = code_dir / "render.py"
    target_base = HF_DATASETS[method_name][dataset_type]["target_dir"]
    
    print(f"\n======================================================")
    print(f" Running FPS Benchmark: {method_name} | Dataset: {dataset_type}")
    print(f"======================================================")
    
    fps_results = {}
    for scene in scenes:
        model_path = find_model_scene_path(target_base, scene)
        if not model_path.exists():
            print(f"[SKIP] Scene directory not found: {model_path}")
            continue
            
        cmd = [sys.executable, str(render_py), "-m", str(model_path), "--skip_train"]
        print(f"\n---> Benchmarking {method_name} | {scene} ...")
        res = subprocess.run(cmd, cwd=str(code_dir), capture_output=True, text=True)
        
        # Parse FPS from results.json
        res_json_path = model_path / "results.json"
        fps_val = None
        if res_json_path.exists():
            try:
                with open(res_json_path, "r") as f:
                    d = json.load(f).get("ours_30000", {})
                    fps_val = d.get("FPS")
            except Exception:
                pass
            
        if fps_val is not None:
            print(f" [RESULT] {scene:12s} FPS: {fps_val:.2f}")
            fps_results[scene] = round(fps_val, 2)
        else:
            print(f" [WARNING] Could not parse FPS for {scene}. Output snippet:")
            print(res.stdout[-300:] if res.stdout else res.stderr[-300:])
            
    return fps_results

print("FPS Benchmark runner registered successfully.")


In [ ]:
all_fps_data = {"mipnerf360": {}, "specular": {}}

methods_list = ["3DGS", "FastGS", "Specular-Gaussians", "Spec-FastGS"]

# 1. Evaluate Mip-NeRF 360 Dataset
print("\n=================== MIP-NERF 360 FPS BENCHMARK ===================")
for method in methods_list:
    all_fps_data["mipnerf360"][method] = run_fps_benchmark(method, "mipnerf360", SCENES_MIPNERF360)

# 2. Evaluate Synthetic Specular Dataset
print("\n=================== SYNTHETIC SPECULAR FPS BENCHMARK ===================")
for method in methods_list:
    all_fps_data["specular"][method] = run_fps_benchmark(method, "specular", SCENES_SPECULAR)


In [ ]:
def build_fps_table(ds_name, scenes):
    rows = []
    for scene in scenes:
        row = {"Scene": scene}
        for method in methods_list:
            row[method] = all_fps_data[ds_name].get(method, {}).get(scene, None)
        rows.append(row)
    df = pd.DataFrame(rows)
    
    # Add Average Row
    avg_row = {"Scene": "Average"}
    for method in methods_list:
        vals = [v for v in df[method] if v is not None]
        avg_row[method] = round(np.mean(vals), 2) if vals else None
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df

df_fps_mip = build_fps_table("mipnerf360", SCENES_MIPNERF360)
df_fps_spec = build_fps_table("specular", SCENES_SPECULAR)

print("\n======================================================")
print("         MIP-NERF 360 RENDERING FPS SUMMARY")
print("======================================================")
display(df_fps_mip)

print("\n======================================================")
print("       SYNTHETIC SPECULAR RENDERING FPS SUMMARY")
print("======================================================")
display(df_fps_spec)

# Export to CSV & Excel in ketqua/
df_fps_mip.to_csv(KETQUA_DIR / "fps_mipnerf360.csv", index=False, encoding="utf-8-sig")
df_fps_spec.to_csv(KETQUA_DIR / "fps_specular.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(KETQUA_DIR / "fps_summary.xlsx") as writer:
    df_fps_mip.to_excel(writer, sheet_name="mipnerf360", index=False)
    df_fps_spec.to_excel(writer, sheet_name="specular", index=False)

print(f"\n[SUCCESS] FPS benchmark tables saved to {KETQUA_DIR.resolve()}")
